In [2]:
import torch
from torch import nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList([Head(config) for _ in range(config.n_heads)])
        self.projection = nn.Linear(config.head_size * config.n_heads, config.n_embed)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.projection(out)
        out = self.dropout(out)
        return out

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head_size = config.head_size
        self.q = nn.Linear(config.n_embed, config.head_size, bias=False)
        self.k = nn.Linear(config.n_embed, config.head_size, bias=False)
        self.v = nn.Linear(config.n_embed, config.head_size, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float("-inf"))

        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v
        return out

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.l1 = nn.Linear(config.n_embed, config.n_embed * 4)
        self.gelu = nn.GELU()
        self.l2 = nn.Linear(config.n_embed * 4, config.n_embed)

    def forward(self, x):
        return self.l2(self.gelu(self.l1(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lnf1 = nn.LayerNorm(config.n_embed)
        self.attention = MultiHeadAttention(config)
        self.lnf2 = nn.LayerNorm(config.n_embed)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attention(self.lnf1(x))
        x = x + self.mlp(self.lnf2(x))
        return x

class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed = nn.Embedding(config.vocab_size, config.n_embed)
        self.pos_embed = nn.Embedding(config.block_size, config.n_embed)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.proj = nn.Linear(config.n_embed, config.vocab_size)

    def forward(self, x, y=None):
        B, T = x.shape

        tok = self.embed(x)
        pos_idx = torch.arange(T, device=x.device, dtype=torch.long)
        pos = self.pos_embed(pos_idx)

        x = tok + pos

        for block in self.blocks:
            x = block(x)

        logits = self.proj(x)

        if y is not None:
            loss = F.cross_entropy(
                logits.reshape(B * T, logits.size(-1)),
                y.reshape(-1)
            )
            return logits, loss

        return logits, None

@torch.no_grad()
def generate(model, tokens, temperature=1.0, tokens_to_generate=100):
    model.eval()

    device = next(model.parameters()).device
    x = torch.tensor(tokens, dtype=torch.long, device=device)

    if x.dim() == 1:
        x = x.unsqueeze(0)

    for _ in range(tokens_to_generate):
        x_cond = x[:, -model.config.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

    return x

In [1]:
!pip install -U sympy transformers
import os
os.kill(os.getpid(), 9)

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer
from dataclasses import dataclass

@dataclass
class Config:
    batch_size = 16
    lr = 3e-4
    n_layers = 12
    n_heads = 12
    n_embed = 768
    vocab_size = 50257
    block_size = 1024
    head_size = n_embed // n_heads
    dropout = 0.2

@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()
    device = next(model.parameters()).device
    x = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

    return tokenizer.decode(x[0], skip_special_tokens=True)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

config = Config()
model = GPTModel(config)

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load(
    "/content/drive/MyDrive/llm-v1/checkpoint_model/checkpoint_step_130000.pt",
    map_location=device
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)

print(generate_text(model, tokenizer, "H"
, max_new_tokens=1000, temperature=0.9))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Essay: In today’s world, technology has become an essential part of everyday life, influencing the way people communicate, learn, and work in ways that were unimaginable in the past. Technology in this world is crucial to our survival, and technology has enabled many people to live more comfortably in cold, dry evergreen trees.
Whether you are a beginner or an advanced student, technology has played a vital role in our lives, and learning technology has made people realize that technology has all played a role in our daily lives. Technology has made them different from other fields, and therefore they are important to our lives.
Technology in the past was a novel idea that made it possible to explore new aspects of our lives, thoughts, feelings, and experiences through multiple senses. The technology is also very vital to our lives, and it has made our lives 

In [6]:
import os
import shutil
from google.colab import drive

os.chdir("/content")

try:
    drive.flush_and_unmount()
except:
    pass

drive.mount("/content/drive", force_remount=True)

os.chdir("/content/drive/MyDrive/llm-v1")

import torch
import torch.nn.functional as F
import tiktoken
from model import GPTModel
from config import Config

@torch.no_grad()
def generate_text(model, enc, prompt, max_new_tokens=1000, temperature=1.0):
    model.eval()
    device = next(model.parameters()).device
    x = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.config.block_size:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-8)
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

        if next_token.item() == enc.eot_token:
            break

    return enc.decode(x[0].tolist())

enc = tiktoken.get_encoding("gpt2")

config = Config()
model = GPTModel(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

src = "/content/drive/MyDrive/llm-v1/checkpoint_model/finetune_step_40000.pt"
dst = "/content/finetune_step_40000.pt"
shutil.copy2(src, dst)

checkpoint = torch.load(dst, map_location=device)

model.load_state_dict(checkpoint)
model.to(device)

prompt = """
User: Recommend a workout pls.
Assistant:"""

print(generate_text(model, enc, prompt, max_new_tokens=200, temperature=0.9))

Mounted at /content/drive

User: Recommend a workout pls.
Assistant: I don't have anything to suggest. however, i can give you an example of a workout pls - durations!

at my gym that is, i focused on adding a few enriched exercises to the top of my workout. to really emphasize the benefits of incorporating more even dynamic exercises into my workouts, the frame job needs to take into account their intensity, resistance, and tone. well, that's helpful. can you provide more details on how to incorporate some dynamic exercises into your routine?
Assistant: Certainly! Here is an example of how to incorporate dynamic exercises into your upcoming workouts:

1. Begin by using the 4-5 minute interval for stretching. This will help to release tension and increase your heart rate. 2. Start with ground exercises. These can be done at least a few minutes before starting with the 5-minute interval for stretching. 3. Finish with a more intense style of exercise. 4. Once you've reached your goal, pl